# Multiclass GP分類とActive Learning：初学者向け実践ガイド

このNotebookでは、**3クラス分類**の学習から、確率予測、Active Learning、q-batch候補生成、Mixed入力まで順に試します。

- `posterior(X)` は**クラスごとの潜在GP**で、分類確率ではありません。
- `predict_proba(X)` は各クラスの事後予測確率です（最後の軸の合計が1）。
- `PredictiveEntropy` は予測ラベルの不確実性、`BALD` はモデルについての情報獲得を重視します。
- `qPredictiveEntropy` は各候補のスコアを集約し、`qBALD` は候補間の依存を含む同時情報量を評価します。

関連：[分類GPの理論](../../docs/theory/24_classification.md) / [分類GPガイド](../../docs/guide/24_classification.md) / [獲得関数対応表](../../docs/models/multiclass-active-learning.md)

## 1. 準備と3クラスの教師データ

ここでは2次元入力からクラス0・1・2を決める小さな人工データを使います。
乱数シードを固定し、再現可能な例にします。

In [ ]:
import torch

from robotorchan.acquisition import (
    BALD,
    PredictiveEntropy,
    qBALD,
    qPredictiveEntropy,
)
from robotorchan.models.classification import (
    MixedMulticlassSingleTaskGPClassifier,
    MulticlassSingleTaskGPClassifier,
)
from robotorchan.optim.strategies.sobol import SobolSearchStrategy

torch.manual_seed(17)
dtype = torch.double
train_X = torch.rand(24, 2, dtype=dtype)
train_Y = torch.bucketize(train_X[:, 0], torch.tensor([0.33, 0.66], dtype=dtype))
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype)
print("入力:", train_X.shape, "ラベル:", train_Y.unique(sorted=True))

## 2. モデルの構築と学習

クラスごとの変分GPを利用します。ここではNotebookを短時間で実行できるよう学習回数を少なくしています。実務では検証データのNLL・Accuracy・校正などを確認し、学習条件を調整してください。

In [ ]:
model = MulticlassSingleTaskGPClassifier(
    train_X, train_Y, num_classes=3, inducing_points=8
)
model.fit(epochs=5, lr=0.03)
test_X = torch.tensor([[0.2, 0.4], [0.5, 0.4], [0.8, 0.4]], dtype=dtype)
probabilities = model.predict_proba(test_X, num_samples=32)
print("クラス確率:", probabilities)
print("予測クラス:", model.predict_class(test_X))
assert probabilities.shape == (3, 3)
assert torch.allclose(probabilities.sum(-1), torch.ones(3, dtype=dtype))

## 3. 確率と潜在値を混同しない

`posterior` は3クラスに対応する潜在関数の分布です。softmax後の確率は `predict_proba` で取得します。

In [ ]:
latent = model.posterior(test_X)
print("潜在GP平均:", latent.mean.shape)
print("クラス確率:", probabilities.shape)
print("予測エントロピー:", model.predictive_entropy(test_X, num_samples=32))

## 4. 単一候補のActive Learning

未観測点のうち、どの点にラベルを付けるとよいかをスコアで比較します。
ここではエントロピーとBALDを使います。両者の順位が異なることがあります。

In [ ]:
pool = torch.rand(16, 2, dtype=dtype)
entropy = PredictiveEntropy(model)
bald = BALD(model, num_samples=16)
entropy_scores = entropy(pool.unsqueeze(-2))
bald_scores = bald(pool.unsqueeze(-2))
print("Entropy候補:", pool[entropy_scores.argmax()])
print("BALD候補:", pool[bald_scores.argmax()])
assert entropy_scores.shape == (16,)
assert bald_scores.shape == (16,)

## 5. q=2の候補をまとめて提案

`SobolSearchStrategy` は既存の獲得関数を受け取り、探索範囲内の候補を返します。
`qBALD` はラベルの組合せを列挙するため、クラス数をC、候補数をqとするとC**qが256以下である必要があります。

In [ ]:
strategy = SobolSearchStrategy(bounds, num_samples=16, seed=17)
result = strategy.optimize(qPredictiveEntropy(model), q=2)
print("qPredictiveEntropy候補:", result.candidates)
print("獲得関数値:", result.acquisition_value)
joint_scores = qBALD(model, num_samples=16)(result.candidates.unsqueeze(0))
print("同じ候補のjoint qBALD:", joint_scores)
assert result.candidates.shape == (2, 2)
assert joint_scores.shape == (1,)

## 6. Mixed入力（連続値＋カテゴリ）

第2列をカテゴリ0・1・2に変更し、`cat_dims=[1]` を指定します。
モデル側のカテゴリ設定と、候補生成側の離散変数指定は別々の責務です。
ここでは合法なカテゴリ候補のスコア評価を確認します。

In [ ]:
mixed_X = train_X.clone()
mixed_X[:, 1] = torch.arange(len(mixed_X)) % 3
mixed_Y = (torch.arange(len(mixed_X)) % 3).long()
mixed_model = MixedMulticlassSingleTaskGPClassifier(
    mixed_X, mixed_Y, num_classes=3, cat_dims=[1], inducing_points=8
)
mixed_model.fit(epochs=5, lr=0.03)
mixed_candidates = torch.tensor(
    [[[0.2, 0.0], [0.7, 1.0]], [[0.4, 2.0], [0.9, 0.0]]], dtype=dtype
)
mixed_scores = qPredictiveEntropy(mixed_model)(mixed_candidates)
print("Mixed q-batch scores:", mixed_scores)
assert mixed_scores.shape == (2,)

## 7. 次の実験につなげる

提案された候補に対して実際にラベルを観測したら、学習データへ追加し、
モデルを再学習して候補提案を繰り返します。

**注意：** このNotebookはAPIと計算経路を理解するための例です。
性能の優劣を示すベンチマークではありません。比較には複数シード、
ラベル予算、ホールドアウトデータ、Accuracy・NLL・Brier scoreなどが必要です。